In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "30-7"
well_name = '30-7a-2'
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
    well_data = project.get_well_data(well_name)

# Water Saturation Estimation

Water saturation estimation is crucial in petrophysics for several reasons:

1. **Hydrocarbon Volume Calculation**: It helps determine the volume of hydrocarbons in place. Accurate water saturation (Sw) values are essential for calculating the original oil in place (OOIP) and original gas in place (OGIP) volumes¹(https://petroshine.com/fluid-saturation/).
2. **Reservoir Characterization**: Understanding the distribution of water saturation helps in characterizing the reservoir, which is vital for planning production strategies and enhancing recovery¹(https://petroshine.com/fluid-saturation/).
3. **Production Forecasting**: Sw values are used in reservoir models to predict future production and to evaluate the economic viability of the reservoir²(https://www.mdpi.com/2077-1312/9/6/666).

### Methods to Estimate Water Saturation

1. **Resistivity Logs**: This is the most common method, where water saturation is estimated using resistivity measurements from well logs. The Archie equation is often used for clean sands, while modified versions like the Waxman-Smits model are used for shaly sands³(https://petrowiki.spe.org/Water_saturation_determination).
2. **Capillary Pressure Measurements**: Laboratory measurements of capillary pressure and corresponding water saturation provide detailed information about the pore structure and fluid distribution³(https://petrowiki.spe.org/Water_saturation_determination).
3. **Core Analysis**: Direct measurement of water saturation from core samples using techniques like the Dean-Stark method³(https://petrowiki.spe.org/Water_saturation_determination).
4. **Nuclear Magnetic Resonance (NMR)**: NMR logging tools can provide estimates of water saturation by measuring the response of hydrogen nuclei in the formation fluids³(https://petrowiki.spe.org/Water_saturation_determination).

This notebook estimates the water saturation based on resistivity log using Normalized Waxman Smits equation.


***
# Log Derived Water Saturation

In [ ]:
from ipywidgets import widgets, interact

from quick_pp.saturation import pickett_plot

focused_data = project.get_all_data().copy()

wells = widgets.SelectMultiple(
    options=['All'] + list(focused_data['WELL_NAME'].unique()),
    value=['All'],
    description='Wells:'
)
m = widgets.FloatSlider(
    value=2,
    min=1,
    max=5,
    step=.1,
    readout_format='.1f'
)
min_rw = widgets.FloatSlider(
    value=.01,
    min=.001,
    max=1.0,
    step=.001,
    readout_format='.3f'
)
min_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.min(),
    min=focused_data.DEPTH.min(),
    max=focused_data.DEPTH.max() - 10,
    step=.1,
    readout_format='.1f'
)
max_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.max(),
    min=focused_data.DEPTH.min() + 10,
    max=focused_data.DEPTH.max(),
    step=.1,
    readout_format='.1f'
)

@interact(wells=wells, m=m, min_rw=min_rw, min_depth=min_depth, max_depth=max_depth)
def param(wells, m, min_rw, min_depth, max_depth):
    if 'All' in wells:
        data = focused_data[(focused_data.DEPTH >= min_depth) & (focused_data.DEPTH <= max_depth)]
    else:
        data = focused_data[(focused_data.WELL_NAME.isin(wells)) & (focused_data.DEPTH >= min_depth) & (focused_data.DEPTH <= max_depth)]
    pickett_plot(data['RT'], data['PHIT'], m=m, min_rw=min_rw, title=f'Pickett Plot for {wells[0]}')

In [ ]:
import numpy as np
from matplotlib import ticker as mticker

from quick_pp.saturation import *
from quick_pp.porosity import *

water_salinity = 100e3
m = 2

temp_grad = estimate_temperature_gradient(well_data['TVD'], 'metric')
rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
b = estimate_b_waxman_smits(temp_grad, rw)
qv = estimate_qv(well_data.VCLAY, well_data.PHIT, cec_clay=.1)
phit_shale = estimate_shale_porosity(well_data.NPHI, well_data.PHIT)
rt_shale = estimate_rt_shale(well_data.RT, well_data.VCLAY)
qvn = estimate_qvn(well_data.VCLAY, well_data.PHIT, phit_shale)

swt_ws = waxman_smits_saturation(well_data['RT'], rw, well_data.PHIE, B=b, Qv=qvn, m=m)
swt_nws = normalized_waxman_smits_saturation(
    well_data.RT, rw, well_data.PHIT, well_data.VCLAY, phit_shale, rt_shale=8, m=m
)

swt_archie = archie_saturation(well_data.RT, rw, well_data.PHIT, m=m)

fig, axes = plt.subplots(3, 1, figsize=(15, 5), sharex=True)
axes[0].plot(well_data['DEPTH'], swt_nws, label='SWT Normalized WS')
axes[0].plot(well_data['DEPTH'], swt_ws, label='SWT WS')
axes[0].plot(well_data['DEPTH'], swt_archie, label='SWT Archie')
axes[0].plot(well_data['DEPTH'], np.ones(len(well_data)), color='black', linestyle='--')
axes[0].set_ylim(0, 2)
axes[0].legend()

axes[1].plot(well_data['DEPTH'], rw, label='RW')
axes[1].set_yscale('log')
axes[1].yaxis.set_minor_formatter(mticker.ScalarFormatter())
axes[1].legend()

axes[2].plot(well_data['DEPTH'], temp_grad, label='Temperature')
axes[2].legend()

fig.tight_layout()

***
# Plot the results

In [ ]:
from quick_pp.plotter.plotter import plotly_log
from quick_pp.core_calibration import *

# Plot individual results
well_data['SWT'] = swt_ws
fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
fig.show(config=dict(scrollZoom=True))

# Apply to all

In [ ]:
from tqdm import tqdm

water_salinity = 100e3
m = 2

for well_name, plot_data in tqdm(df.groupby('WELL_NAME')):
    tqdm.write(f'Processing {well_name}: {len(plot_data)} rows')

    temp_grad = estimate_temperature_gradient(plot_data['TVD'], 'metric')
    rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
    b = estimate_b_waxman_smits(temp_grad, rw)
    phit_shale = estimate_shale_porosity(plot_data.NPHI, plot_data.PHIT)
    qvn = estimate_qvn(plot_data.VCLAY, plot_data.PHIT, phit_shale)
    swt = waxman_smits_saturation(plot_data['RT'], rw, plot_data.PHIE, B=b, Qv=qvn, m=m)

    plot_data['SWT'] = swt.clip(0, 1)
    
    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(plot_data)
        project.save()